<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/07_name_entity_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Named Entity Recognition (NER) in spaCy

## 1. What is NER? (Easy Explanation)

NER means spaCy scans your text and picks out **real-world "things"** — people, places, companies, dates, money — and labels each one.

```
"Tesla Inc is going to acquire twitter for $45 billion"
   |__________|                              |_________|
     ORG (company)                             MONEY (amount)
```

You don't need to tell spaCy what to look for — the trained pipeline (`en_core_web_sm`) already knows common entity types out of the box.

## 2. Basic NER — Finding Entities

In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")
nlp.pipe_names   # 'ner' is one of the components -> this is what powers entity detection


['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [2]:
doc = nlp("Tesla Inc is going to acquire twitter for $45 billion")

for ent in doc.ents:
    print(ent.text, " | ", ent.label_, " | ", spacy.explain(ent.label_))


Tesla Inc  |  ORG  |  Companies, agencies, institutions, etc.
$45 billion  |  MONEY  |  Monetary values, including unit


## 3. Visualizing Entities — `displacy`

Instead of reading plain text output, you can highlight entities directly in the notebook with colors.

In [3]:
from spacy import displacy

displacy.render(doc, style="ent")


## 4. All Entity Types spaCy Knows

You don't have to memorize labels like `ORG`, `GPE`, `MONEY` — spaCy can list them all for you, and `spacy.explain()` tells you what each one means.

In [4]:
nlp.pipe_labels["ner"]


['CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART']

**Output:**
```
['CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY',
 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART']
```
Full docs: https://spacy.io/models/en

A few common ones to remember:
- **PERSON** → people
- **GPE** → countries, cities, states
- **ORG** → companies, institutions
- **DATE** → dates or periods
- **MONEY** → money amounts

In [5]:
doc = nlp("Michael Bloomberg founded Bloomberg in 1982")
for ent in doc.ents:
    print(ent.text, "|", ent.label_, "|", spacy.explain(ent.label_))


Michael Bloomberg | PERSON | People, including fictional
Bloomberg | GPE | Countries, cities, states
1982 | DATE | Absolute or relative dates or periods




**Important:** NER is NOT perfect. Here it wrongly tagged "Bloomberg" (the company) as `GPE` (a place), just because "Bloomberg" is also a well-known city-sounding name. Small models like `en_core_web_sm` make mistakes like this — bigger models or specialized models (e.g. HuggingFace's `dslim/bert-base-NER`) can do better in tricky cases.

## 5. Getting Character Positions of Entities

Sometimes you need to know *where exactly* (character index) an entity starts and ends in the text — useful for highlighting text in a UI, for example.

In [6]:
doc = nlp("Tesla Inc is going to acquire Twitter Inc for $45 billion")

for ent in doc.ents:
    print(ent.text, " | ", ent.label_, " | ", ent.start_char, "|", ent.end_char)


Tesla Inc  |  ORG  |  0 | 9
Twitter Inc  |  PERSON  |  30 | 41
$45 billion  |  MONEY  |  46 | 57


## 6. Setting Custom Entities Manually

If spaCy misses an entity or labels it wrong, you can manually override it using `Span`.

In [7]:
doc = nlp("Tesla is going to acquire Twitter for $45 billion")

for ent in doc.ents:
    print(ent.text, " | ", ent.label_)


Tesla  |  ORG
Twitter  |  PERSON
$45 billion  |  MONEY


**Output (before fixing):**
```
Twitter      |  PRODUCT/PERSON     <- "Tesla" got missed entirely, "Twitter" mislabeled as PRODUCT/PERSON
$45 billion  |  MONEY
```

In [8]:
# check token positions first (index-based slicing)
s = doc[2:5]
s


going to acquire

In [9]:
from spacy.tokens import Span

# manually define spans: doc[0:1] = "Tesla", doc[5:6] = "Twitter", force label to ORG
s1 = Span(doc, 0, 1, label="ORG")
s2 = Span(doc, 5, 6, label="ORG")

doc.set_ents([s1, s2], default="unmodified")   # apply our custom entities, keep everything else as-is


In [10]:
for ent in doc.ents:
    print(ent.text, " | ", ent.label_)


Tesla  |  ORG
Twitter  |  ORG
$45 billion  |  MONEY


**Output (after fixing):**
```
Tesla        |  ORG
Twitter      |  ORG
$45 billion  |  MONEY
```
Now both companies are correctly tagged as `ORG`.

---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Get entities | `doc.ents` |
| Entity text/label | `ent.text`, `ent.label_` |
| Explain a label | `spacy.explain(label)` |
| Visualize entities | `displacy.render(doc, style="ent")` |
| List all entity types | `nlp.pipe_labels["ner"]` |
| Entity character position | `ent.start_char`, `ent.end_char` |
| Manually set entities | `Span(doc, start, end, label="X")` + `doc.set_ents([...])` |

---

## Exercise 1 — Extract Geographical Names (Cities, Countries, States)

**Task:** From the text below, extract all geographical location names, and count them.

In [11]:
text = """Kiran want to know the famous foods in each state of India. So, he opened Google and search for this question. Google showed that
in Delhi it is Chaat, in Gujarat it is Dal Dhokli, in Tamilnadu it is Pongal, in Andhrapradesh it is Biryani, in Assam it is Papaya Khar,
in Bihar it is Litti Chowkha and so on for all other states"""

doc = nlp(text)

geographical_names = []

for ent in doc.ents:
    if ent.label_ == "GPE":   # GPE = Geo-Political Entity -> countries, cities, states
        geographical_names.append(ent)

print("Geographical location Names: ", geographical_names)
print("Count: ", len(geographical_names))


Geographical location Names:  [Kiran, India, Delhi, Gujarat, Tamilnadu, Andhrapradesh, Assam, Bihar]
Count:  8


## Exercise 2 — Extract Birth Dates of Cricketers

**Task:** From the text below, extract all the dates mentioned, and count them.

In [12]:
text = """Sachin Tendulkar was born on 24 April 1973, Virat Kholi was born on 5 November 1988, Dhoni was born on 7 July 1981
and finally Ricky ponting was born on 19 December 1974."""

doc = nlp(text)

all_birth_dates = []

for ent in doc.ents:
    if ent.label_ == "DATE":   # DATE = absolute or relative dates/periods
        all_birth_dates.append(ent)

print("All Birth Dates: ", all_birth_dates)
print("Count: ", len(all_birth_dates))


All Birth Dates:  [24 April 1973, 5 November 1988, 7 July 1981, 19 December 1974]
Count:  4
